# Returns Forecasting: Training Pipeline

This notebook trains two forecasting models for portfolio optimization:

1. **LightGBM** — predicts expected returns per asset (replaces historical mean)
2. **GARCH** — forecasts the covariance matrix (replaces historical covariance)

Both models are exported for deployment to CAII (Cloudera AI Inference Services).

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portfolio_optimization.utils import download_data, get_input_data, calculate_returns
from portfolio_optimization.settings import ReturnsComputeSettings
from portfolio_optimization.forecasting.config import ForecastingConfig
from portfolio_optimization.forecasting.feature_engineering import (
    compute_features, build_training_data, flatten_features_for_training
)
from portfolio_optimization.forecasting.lightgbm_model import ReturnsForecaster
from portfolio_optimization.forecasting.garch_model import CovarianceForecaster
from portfolio_optimization.forecasting.caii_client import ForecastClient

print("Imports OK")

In [ ]:
# Download DOW30 data if not already present
data_dir = "../data/stock_data"
download_data(data_dir, datasets=["dow30"])

prices = get_input_data(f"{data_dir}/dow30.csv")
print(f"Price data: {prices.shape[0]} days x {prices.shape[1]} assets")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
prices.tail()

## 2. Configuration

Key parameters:
- `forecast_horizon`: days ahead to predict (5 = weekly)
- `training_window`: lookback days for training (~756 = 3 years)
- Feature windows for momentum, volatility, RSI

In [ ]:
config = ForecastingConfig(
    forecast_horizon=5,
    training_window=756,
    return_type="LOG",
)
print(config.model_dump_json(indent=2))

## 3. Feature Engineering

Per-asset features: momentum, rolling volatility, RSI, price vs SMA, cross-asset correlation.

In [ ]:
features = compute_features(prices, config.features)
print(f"Feature matrix: {features.shape}")
print(f"\nFeatures per asset:")
sample_ticker = prices.columns[0]
print([col for col in features[sample_ticker].columns])

In [ ]:
X, y = build_training_data(
    prices,
    forecast_horizon=config.forecast_horizon,
    return_type=config.return_type,
    config=config.features,
)
X_flat, y_flat = flatten_features_for_training(X, y)
print(f"Flattened training set: {X_flat.shape[0]} samples x {X_flat.shape[1]} features")
print(f"Target distribution: mean={y_flat.mean():.6f}, std={y_flat.std():.4f}")

## 4. Train LightGBM (Expected Returns)

In [ ]:
returns_forecaster = ReturnsForecaster(config)
metrics = returns_forecaster.train(prices)
print(f"\nTraining metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

In [ ]:
importance = returns_forecaster.feature_importance()
print("Top 10 features by importance:")
print(importance.head(10).to_string(index=False))

In [ ]:
# Predict expected returns using the most recent data
predicted_returns = returns_forecaster.predict(prices)
print("Predicted expected returns (next 5 days):")
for ticker, ret in zip(prices.columns, predicted_returns):
    print(f"  {ticker}: {ret:+.6f}")

## 5. Train GARCH (Covariance Forecasting)

In [ ]:
cov_forecaster = CovarianceForecaster(config)
garch_metrics = cov_forecaster.train(prices)

converged = sum(1 for m in garch_metrics.values() if m.get("converged"))
print(f"GARCH models converged: {converged}/{len(garch_metrics)}")
print(f"\nSample metrics (first 5 assets):")
for ticker in list(garch_metrics.keys())[:5]:
    m = garch_metrics[ticker]
    if m.get("converged"):
        print(f"  {ticker}: AIC={m['aic']:.1f}, persistence={m['persistence']:.4f}")
    else:
        print(f"  {ticker}: FAILED - {m.get('error', 'unknown')}")

In [ ]:
# Forecast covariance matrix
predicted_cov = cov_forecaster.predict(prices)
predicted_vol = np.sqrt(np.diag(predicted_cov))
print("Predicted annualized volatility (top 5):")
vol_series = pd.Series(predicted_vol * np.sqrt(252), index=prices.columns)
print(vol_series.sort_values(ascending=False).head().to_string())

## 6. Compare: Historical vs. Forecasted Inputs

In [ ]:
# Historical baseline
returns_dict = calculate_returns(
    prices,
    returns_compute_settings=ReturnsComputeSettings(return_type="LOG", freq=1),
)

hist_mean = returns_dict["mean"]
hist_cov = returns_dict["covariance"]

print("Mean return comparison (annualized):")
comparison = pd.DataFrame({
    "Historical": hist_mean * 252,
    "LightGBM Forecast": predicted_returns * 252 / config.forecast_horizon,
}, index=prices.columns)
comparison["Difference"] = comparison["LightGBM Forecast"] - comparison["Historical"]
print(comparison.sort_values("Difference", ascending=False).head(10).to_string())

## 7. Export Models

- LightGBM → ONNX format for CAII deployment
- GARCH → JSON parameters (GARCH is recursive, not natively ONNX)

In [ ]:
import os
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

# Save LightGBM (native format — always works)
lgb_path = returns_forecaster.save(f"{model_dir}/returns_forecaster.lgb")
print(f"LightGBM saved: {lgb_path}")

# Export LightGBM to ONNX (requires onnxmltools)
try:
    onnx_path = returns_forecaster.export_onnx(f"{model_dir}/returns_forecaster.onnx")
    print(f"ONNX exported: {onnx_path}")
except ImportError:
    print("Install onnxmltools and skl2onnx for ONNX export:")
    print("  pip install onnxmltools skl2onnx")

# Export GARCH parameters
garch_path = cov_forecaster.export_params(f"{model_dir}/garch_params.json")
print(f"GARCH params saved: {garch_path}")

## 7b. Register in MLflow (for CAII Deployment)

MLflow model registry is the bridge to CAII — register the ONNX model so CAII can deploy it as an inference endpoint.

In [ ]:
# Register LightGBM returns forecaster in MLflow → CAII can deploy this
returns_uri = returns_forecaster.register_mlflow(
    model_name="PortfolioReturnsForecaster"
)
print(f"Returns model URI: {returns_uri}")

# Register GARCH covariance forecaster params in MLflow
cov_uri = cov_forecaster.register_mlflow(
    model_name="PortfolioCovarianceForecaster"
)
print(f"Covariance model URI: {cov_uri}")

## 8. Integration with Optimizer

Use `ForecastClient.update_returns_dict()` to swap in forecasted values before optimization.

In [ ]:
# Create forecast client with local models
client = ForecastClient(
    config=config,
    returns_model=returns_forecaster,
    covariance_model=cov_forecaster,
)

# Get returns_dict with historical values, then replace with forecasts
returns_dict_forecasted = client.update_returns_dict(returns_dict, prices)

print("returns_dict updated with forecasted values.")
print(f"  mean shape: {returns_dict_forecasted['mean'].shape}")
print(f"  covariance shape: {returns_dict_forecasted['covariance'].shape}")
print("\nReady for optimizer: pass returns_dict_forecasted to CVaR or MeanVariance.")